# Specs — A-Class EDA

Profiles `data/external/mercedes_specs_A_class.csv` (scraped spec sheet, one row per A-Class variant) before it is used to enrich the resale pool.

Focus: **(1) duplicate rows** and **(2) rows with too much missing data**.

In [ ]:
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = pathlib.Path.cwd()
while not (ROOT / 'data' / 'external').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CSV_PATH = ROOT / 'data' / 'external' / 'mercedes_specs_A_class.csv'

sns.set_theme()
pd.set_option('display.max_columns', 60)

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

## Shape & dtypes

In [ ]:
df.info()

## Duplicate rows

Three lenses, from strictest to most meaningful:
1. **Exact** full-row duplicates.
2. Duplicates ignoring `source_url` / `scraped_at` — these two columns are scrape metadata and will always be unique even if every spec value is identical.
3. Duplicates on the **business key** (`version_name`, `year_start`, `year_end`) — the same trim/year combination scraped more than once.

In [ ]:
print('exact duplicate rows:', df.duplicated().sum())

meta_cols = ['source_url', 'scraped_at']
spec_cols = [c for c in df.columns if c not in meta_cols]
spec_dupes = df.duplicated(subset=spec_cols, keep=False)
print('duplicate rows ignoring source_url/scraped_at:', spec_dupes.sum())
df.loc[spec_dupes].sort_values('version_name')

In [ ]:
key_cols = ['version_name', 'year_start', 'year_end']
key_dupes = df.duplicated(subset=key_cols, keep=False)
print('duplicate rows on (version_name, year_start, year_end):', key_dupes.sum())
df.loc[key_dupes, key_cols].sort_values(key_cols)

## Missing values — by column

Which fields are sparsest. Expect the EV/hybrid-only fields (`battery_*`, `charging_*`, `total_electric_*`) to be mostly blank since most A-Class variants are pure ICE.

In [ ]:
col_missing = (df.isna().mean() * 100).sort_values(ascending=False)
col_missing = col_missing[col_missing > 0]
print(f'{len(col_missing)} of {df.shape[1]} columns have at least one missing value')
col_missing

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
col_missing.plot.barh(ax=ax)
ax.set_xlabel('% missing')
ax.invert_yaxis()
plt.tight_layout()

## Missing values — by row

How much of *each row* is missing, and how many rows cross a "too much missing data" threshold. Adjust `THRESHOLD` below to test a stricter or looser cut.

In [ ]:
row_missing_pct = df.isna().mean(axis=1) * 100
print(row_missing_pct.describe())

fig, ax = plt.subplots(figsize=(7, 4))
row_missing_pct.plot.hist(bins=20, ax=ax)
ax.set_xlabel('% of columns missing per row')
plt.tight_layout()

In [ ]:
for threshold in (30, 50, 70):
    n = (row_missing_pct > threshold).sum()
    print(f'rows with >{threshold}% missing: {n} ({n / len(df) * 100:.1f}%)')

THRESHOLD = 50
flagged = df.loc[row_missing_pct > THRESHOLD]
print(f'\nrows flagged at >{THRESHOLD}% missing: {len(flagged)}')
flagged

## Summary

- **Duplicates:** none found under any of the three checks above — each row is a distinct `version_name` + year-range combination.
- **Missingness:** column-level gaps are concentrated in EV/hybrid-only fields, which is expected for ICE variants rather than a data-quality problem. Row-level missing % stays well clear of a 50% threshold, so no rows currently need to be dropped for missingness alone — re-run after the next scrape to confirm this still holds.